# 03. PromptTemplate과 Message

PromptTemplate을 사용하면 프롬프트를 재사용 가능한 템플릿으로 관리할 수 있습니다.

In [1]:
import os
import sys
from dotenv import load_dotenv
sys.path.append("..")  # 상위 폴더의 common_config.py를 import하기 위해 추가
from common_config import get_llm

load_dotenv(override=True, dotenv_path="../.env")

True

## 1. PromptTemplate - 단일 문자열 템플릿

In [2]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template("{topic}에 대해 한 문장으로 설명해줘")

# 변수 채우기
prompt = template.invoke({"topic": "RAG"})
print(prompt)

text='RAG에 대해 한 문장으로 설명해줘'


## 2. ChatPromptTemplate - 역할 기반 템플릿

시스템/사용자/AI 메시지를 조합하여 템플릿을 만듭니다.

In [3]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role} 전문가입니다."),
    ("human", "{question}")
])

messages = chat_prompt.invoke({
    "role": "파이썬",
    "question": "데코레이터가 뭔가요?"
})

print(messages)

messages=[SystemMessage(content='당신은 파이썬 전문가입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='데코레이터가 뭔가요?', additional_kwargs={}, response_metadata={})]


## 3. LLM과 연결하여 호출

In [5]:
from langchain_openai import ChatOpenAI

llm = get_llm("gpt-5.4-mini", temperature=0)

messages = chat_prompt.invoke({
    "role": "머신러닝",
    "question": "과적합이 뭔가요?"
})

response = llm.invoke(messages)
print(response.content)

과적합(overfitting)은 **모델이 학습 데이터에 너무 딱 맞게 학습되어, 새로운 데이터에는 잘 못 맞추는 현상**입니다.

### 쉽게 말하면
시험 문제를 **외워서** 풀었는데,  
시험지가 조금만 바뀌면 못 푸는 상태와 비슷합니다.

### 왜 생기나요?
- 모델이 너무 복잡할 때
- 학습 데이터가 너무 적을 때
- 노이즈(우연한 값)까지 학습해버릴 때

### 과적합의 특징
- **훈련 데이터 성능은 매우 좋음**
- **검증/테스트 데이터 성능은 나쁨**

즉, **겉보기에는 잘 맞지만 일반화가 안 되는 상태**입니다.

### 예시
스팸 메일 분류기를 만들었는데,
학습 메일에만 나온 특정 단어를 너무 중요하게 외워서  
새로운 스팸 메일은 잘 못 잡는 경우입니다.

### 막는 방법
- 더 많은 데이터 사용
- 모델 단순화
- 정규화(regularization) 사용
- 드롭아웃(dropout)
- 조기 종료(early stopping)
- 데이터 증강

원하시면 제가 **과적합, 과소적합, 일반화**를 한 번에 비교해서 설명해드릴게요.


## 4. 대화 히스토리 포함 - MessagesPlaceholder

이전 대화 내용을 프롬프트에 삽입할 때 사용합니다.

In [6]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 AI 어시스턴트입니다."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

# 이전 대화 기록
history = [
    HumanMessage(content="내 이름은 홍길동이야"),
    AIMessage(content="안녕하세요, 홍길동님!")
]

messages = chat_prompt.invoke({
    "history": history,
    "question": "내 이름이 뭐라고?"
})

response = llm.invoke(messages)
print(response.content)

홍길동입니다.


In [7]:
# 현재 대화를 history에 추가
history.append(HumanMessage(content="내 이름이 뭐라고?"))
history.append(AIMessage(content=response.content))
history

[HumanMessage(content='내 이름은 홍길동이야', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, 홍길동님!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='내 이름이 뭐라고?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='홍길동입니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

## 5. 다중 변수 템플릿 예제

In [16]:
template = ChatPromptTemplate.from_messages([
    ("system", "당신은 {language} 번역가입니다."),
    ("human", "다음 문장을 번역해줘: {sentence}")
])

messages = template.invoke({
    "language": "중국어",
    "sentence": "习近平和金正恩被描述为一种有争议的关系。"
})

response = llm.invoke(messages)
print(response.content)

시진핑과 김정은은 논란의 여지가 있는 관계로 묘사된다.


## 문제 해결

In [9]:
template = ChatPromptTemplate.from_messages([
    ("system", """당신은 {sorted_language}에서 {target_language}로 번역하는 번역가입니다.
      출력을 할 때 {sorted_language}로 번역된 문장과 {target_language}로 번역된 문장을 모두 보여주세요.
     
      예시:
      입력(한국어): "오늘 날씨가 정말 좋네요.
      출력(영어): The weather is really nice today. 
     """),
    ("human", "다음 문장을 번역해줘: {text}")
])

messages = template.invoke({
    "sorted_language": "한국어",
    "target_language": "영어",
    "text": "오늘 날씨가 정말 좋네요."
})

response = llm.invoke(messages)
print(response.content)

한국어: 오늘 날씨가 정말 좋네요.  
영어: The weather is really nice today.
